# 雪乃离线对话 · A100 多轮训练

在 Colab 的「运行时 → 更改运行时类型」选择 A100 GPU，再点「全部运行」。按顺序执行第 1–6 节。首次运行会验证 Drive 上的 20 册 EPUB 和固定源码。

本次沿用 Qwen2.5-1.5B-Instruct，默认先训练 **4 轮人物知识**，再用同一适配器续训 **3 轮雪乃人格**。一轮是遍历一次相应的训练集。每轮结束评估并保存检查点，默认保留这 7 份检查点。轮数、日志、数据和权重都能在云盘中核对。

本版使用新的运行目录 `epub-a100-v2`。同一运行目录一次只运行一份笔记本。中断后重连 A100，再用原 `RUN_ID` 和原轮数重新运行全部单元。程序会跳过已完成阶段，并从最近一个可恢复的检查点继续未完成阶段。

## 1. 核验 A100 并挂载 Google Drive

先检查实际分配的 GPU，再确认 Drive 中有 20 册 EPUB 和对话样例。设置运行名称和两阶段总轮数。若要改轮数，请在启动新运行前修改，并使用新的 `RUN_ID`。

In [ ]:
# 1. 确认实际分配到 A100，挂载 Drive，设置本次试训名称
import json
import re
import subprocess
import sys
from pathlib import Path

from google.colab import drive

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
).strip()
print(gpu)
if "A100" not in gpu:
    raise RuntimeError("当前运行时未分配 A100，请先更改运行时类型。")
drive.mount("/content/drive")

RUN_ID = "epub-a100-v2"  # 改轮数、数据或重开实验时使用新名称
KNOWLEDGE_EPOCHS = 4
PERSONA_EPOCHS = 3
if min(KNOWLEDGE_EPOCHS, PERSONA_EPOCHS) < 2:
    raise ValueError("两阶段的训练轮数都必须至少为 2。")
CHECKPOINT_KEEP = max(KNOWLEDGE_EPOCHS, PERSONA_EPOCHS)
SCENARIO_FILE = "scenarios.jsonl"  # 更换成 data/ 下另一份已审阅的文件名
KNOWLEDGE_ATTEMPT = 1  # 没有可恢复检查点时，可增加编号重新运行该阶段
PERSONA_ATTEMPT = 1  # 继续训练中途失败时，增加编号重新运行人格阶段
PRIVATE_TEST_FILE = ""  # 可填 data/ 下另一份独立题集的文件名
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", RUN_ID):
    raise ValueError("RUN_ID 只能使用字母、数字、连字符和下划线。")

root = Path("/content/yukino-a100-v2")
root.mkdir(exist_ok=True)
project = Path("/content/drive/MyDrive/Yukino-Companion")
data_root = project / "data"
drive_runs = project / "retrain" / RUN_ID
for name in (SCENARIO_FILE, PRIVATE_TEST_FILE):
    if name and (Path(name).name != name or not name.endswith(".jsonl")):
        raise ValueError("数据配置只能填 data/ 目录下的 .jsonl 文件名。")
epubs = sorted((data_root / "epubs").glob("*.epub"))
if not epubs or any(not re.match(r"^\[\d+(?:\.\d+)?\]", path.name) for path in epubs):
    raise ValueError("请检查 data/epubs/ 中的 EPUB，文件名须以 [卷号] 开头。")
source = data_root / SCENARIO_FILE
challenge = data_root / "challenge-prompts.jsonl"
for required in (source, challenge):
    if not required.is_file():
        raise FileNotFoundError(required)
drive_runs.mkdir(parents=True, exist_ok=True)
print("原作 EPUB 数量", len(epubs))
print("输出目录", drive_runs)

## 2. 恢复固定版本源码与训练环境

校验并读取 Drive 上的源码归档，复用 Colab 自带的 CUDA PyTorch，然后安装固定版本的其他训练依赖。验证 A100 支持 BF16，检查 Qwen 底座的 SHA-256。安装日志和实际依赖版本留在 Drive。

In [ ]:
# 2. 安装训练环境，下载并验证固定底座
import shutil
import tempfile
from hashlib import file_digest

def digest(path):
    with Path(path).open("rb") as stream:
        return file_digest(stream, "sha256").hexdigest()

from zipfile import ZipFile
import os
import tomllib

repo = root / "source"
commit = "e846aa4c32cc2cc657e05ddc839372690bdbf18b"
source_archive = project / "source" / "training-source-e846aa4c-checkpoints-v2.zip"
source_hash = "680a4d6ccb07b6b668fee8606435148b14137f4826f47e5befd8c5ba4f5d9a20"
if not source_archive.is_file() or digest(source_archive) != source_hash:
    raise RuntimeError(f"Drive 中的固定源码归档缺失或校验失败：{source_archive}")
with ZipFile(source_archive) as archive:
    if archive.comment.decode("ascii") != commit:
        raise RuntimeError("源码归档的 Git 提交与笔记本记录不一致。")
    if not repo.exists():
        with tempfile.TemporaryDirectory(prefix="source-", dir=root) as work:
            restored = Path(work) / "source"
            archive.extractall(restored)
            restored.rename(repo)
    else:
        for member in archive.infolist():
            if member.is_dir():
                continue
            local_file = repo / member.filename
            if not local_file.is_file() or local_file.read_bytes() != archive.read(member):
                raise RuntimeError(f"本地源码缓存与 Drive 归档不一致：{member.filename}。请重置 Colab 运行时后重试。")
print("Drive 源码", source_archive)

service = repo / "model-service"
venv = root / "venv-colab-cuda"
python = venv / "bin/python"
if not python.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(venv)], check=True)

def run(*args, cwd=service, log=None):
    command = [str(python), *(str(arg) for arg in args)]
    env = dict(os.environ, USE_TF="0", USE_TORCH="1")
    if log is None:
        subprocess.run(command, cwd=cwd, env=env, check=True)
        return
    with log.open("a", encoding="utf-8", buffering=1) as stream:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                              errors="replace", bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)

setup_log = drive_runs / "setup-colab-cuda.log"
project_config = tomllib.loads((service / "pyproject.toml").read_text(encoding="utf-8"))["project"]
dependencies = project_config["dependencies"] + [
    spec for spec in project_config["optional-dependencies"]["train"]
    if not spec.startswith("torch")
]
torch_check = (
    "import sys, torch; "
    "print(sys.version, torch.__version__, torch.version.cuda, flush=True); "
    "assert tuple(map(int, torch.__version__.split('.')[:2])) >= (2, 6); "
    "assert 'A100' in torch.cuda.get_device_name(0) and torch.cuda.is_bf16_supported(); "
    "print(torch.ones(1, device='cuda', dtype=torch.bfloat16))"
)
run("-c", torch_check, log=setup_log)

deps_ready = venv / "yukino-deps-ready"
if not deps_ready.exists():
    run("-m", "pip", "install", "--disable-pip-version-check", "--retries", "3",
        "--timeout", "60", *dependencies, log=setup_log)
    run("-m", "pip", "install", "--no-deps", "-e", ".", log=setup_log)

run("-c", "from transformers import Trainer; from peft import LoraConfig; import yuki_model", log=setup_log)
deps_ready.touch()
run("-m", "pip", "list", "--format=json", log=drive_runs / "environment-colab-cuda.log")

base = Path("/content/yukino-a100/source/model-service/models/base")
if not (base / "BASE_PROVENANCE.json").is_file():
    run("-m", "yuki_model", "download", "--output", base, log=setup_log)
base_revision = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
expected_base_weights = "dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee"
base_proof = json.loads((base / "BASE_PROVENANCE.json").read_text(encoding="utf-8"))
if base_proof["base_revision"] != base_revision or digest(base / "model.safetensors") != expected_base_weights:
    raise RuntimeError("基础模型权重与固定版本不符。")

## 3. 从 EPUB 准备人物知识和对话训练数据

从小说人物介绍构建有出处的问答，并生成原作章节索引。人格对话按场景族切分训练、验证和测试集。复制到 Drive 后逐文件核验，已有相同数据可直接复用。

In [ ]:
# 3. 从私有 EPUB 构建原作索引和人物事实 SFT，再构建人格数据
def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def save_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)

def inventory(directory):
    return {
        path.relative_to(directory).as_posix(): digest(path)
        for path in sorted(directory.rglob("*")) if path.is_file() and path.name != "artifacts.json"
    }

def publish_directory(local, final):
    with tempfile.TemporaryDirectory(prefix=".publish-", dir=final.parent) as staging:
        candidate = Path(staging) / final.name
        shutil.copytree(local, candidate)
        if inventory(candidate) != inventory(local):
            raise RuntimeError("复制到 Drive 后的文件校验失败。")
        candidate.rename(final)

epub_hashes = {book.name: digest(book) for book in epubs}
inputs = {
    "source_commit": commit,
    "source_archive_sha256": source_hash,
    "base_revision": base_revision,
    "epubs": epub_hashes,
    "persona_source_sha256": digest(source),
}
inputs_path = drive_runs / "inputs.json"
if inputs_path.exists():
    if load_json(inputs_path) != inputs:
        raise ValueError("这个 RUN_ID 的训练输入已有不同快照，请换新名称。")
else:
    save_json(inputs_path, inputs)

prepared = drive_runs / "prepared"
if not prepared.exists():
    with tempfile.TemporaryDirectory(prefix="prepare-", dir=root) as work:
        work = Path(work)
        local_epubs = work / "epubs"
        local_epubs.mkdir()
        for book in epubs:
            copied = local_epubs / book.name
            shutil.copyfile(book, copied)
            if digest(copied) != epub_hashes[book.name]:
                raise RuntimeError(f"复制期间 EPUB 内容发生变化：{book.name}")
        candidate = work / "data"
        processed = candidate / "processed"
        run("-m", "yuki_model", "prepare", "--input", local_epubs, "--output", processed)
        run("-m", "yuki_model", "index", "--input", processed / "chunks.jsonl",
            "--output", processed / "knowledge.sqlite")
        try:
            # build_sft.py reads data/processed and writes data/sft relative to its working directory.
            run(service / "scripts/build_sft.py", cwd=work)
        except subprocess.CalledProcessError as error:
            raise RuntimeError("人物卡 SFT 构建失败。请检查 EPUB 是否使用仓库支持的中文人物介绍格式。") from error
        persona = candidate / "persona"
        persona.mkdir()
        copied_source = persona / "scenarios.jsonl"
        shutil.copyfile(source, copied_source)
        if digest(copied_source) != inputs["persona_source_sha256"]:
            raise RuntimeError("复制期间人格数据发生变化。")
        run("scripts/build_persona_sft.py", "--source", copied_source,
            "--tokenizer", base, "--output", persona)

        # Same questions are regression probes on known facts; references are excluded from the prompts.
        heldout = candidate / "sft/known-fact-prompts.jsonl"
        references = candidate / "sft/known-fact-references.jsonl"
        with (candidate / "sft/val.jsonl").open(encoding="utf-8") as source_rows, \
                heldout.open("w", encoding="utf-8") as prompts, references.open("w", encoding="utf-8") as answers:
            for number, line in enumerate(source_rows, 1):
                row = json.loads(line)
                family = f"known_fact_{number:03d}"
                prompt = {
                    "family_id": family,
                    "category": row["kind"],
                    "variant_id": "validation",
                    "messages": [
                        {"role": "system", "content": "评估时使用统一人物提示。"},
                        row["messages"][-2],
                    ],
                }
                reference = {"family_id": family, "source_ids": row["source_ids"], "answer": row["messages"][-1]["content"]}
                prompts.write(json.dumps(prompt, ensure_ascii=False) + "\n")
                answers.write(json.dumps(reference, ensure_ascii=False) + "\n")

        save_json(candidate / "artifacts.json", inventory(candidate))
        publish_directory(candidate, prepared)

if inventory(prepared) != load_json(prepared / "artifacts.json"):
    raise RuntimeError("Drive 中的训练数据与已保存的摘要不一致。")
processed, knowledge_data, persona_data = (prepared / name for name in ("processed", "sft", "persona"))
corpus = load_json(processed / "manifest.json")
if {book["file"]: book["sha256"] for book in corpus["books"]} != epub_hashes:
    raise RuntimeError("索引的 EPUB 来源与本次输入不一致。")
print("原作章节", corpus["sections"], "文本块", corpus["chunks"])
print("知识 SFT", load_json(knowledge_data / "manifest.json"))
print("人格 SFT", load_json(persona_data / "manifest.json")["examples"])

## 4. 训练人物知识，再续训雪乃对话

知识阶段默认 4 轮，人格阶段默认 3 轮。两阶段每轮结束都保存 LoRA 权重、优化器、学习率调度、随机状态和训练步数。完成后的适配器和训练报告也单独保存在 Drive。

重连时自动挑选同一阶段里最新的完整检查点。如果在第一轮结束前中断，增加对应的 `KNOWLEDGE_ATTEMPT` 或 `PERSONA_ATTEMPT` 编号，给该阶段开一个新的输出目录。

In [ ]:
# 4. 训练知识 LoRA，再从本轮知识权重继续训练人格 LoRA
RUN_TRAINING = True  # 只想先检查数据时改为 False

def training_config(filename, data, output, **options):
    config = load_json(service / "configs" / filename)
    config.update(
        model=str(base), device="cuda", allow_cpu=False, quantize_4bit=False,
        train_file=str(data / "train.jsonl"), eval_file=str(data / "val.jsonl"),
        output_dir=str(output), save_total_limit=CHECKPOINT_KEEP, **options,
    )
    return config

def verify_run(output, task):
    report = load_json(output / "training_report.json")
    if not (report["completed"] and report["run_purpose"] == "training"
            and report["task"] == task and report["steps"] > 1
            and report["base_revision"] == base_revision and report["device"] == "cuda"
            and report["dtype"] == "torch.bfloat16"
            and report["adapter_sha256"] == digest(output / "adapter/adapter_model.safetensors")):
        raise RuntimeError(f"训练报告或权重校验失败：{output}")
    return report

def checkpoint_state(path):
    try:
        state = load_json(path / "trainer_state.json")
        return int(state["global_step"]), float(state["epoch"])
    except (OSError, ValueError, KeyError, TypeError):
        return None

def complete_checkpoints(output):
    required = ("adapter_config.json", "adapter_model.safetensors", "trainer_state.json",
                "optimizer.pt", "scheduler.pt", "rng_state.pth")
    found = []
    for path in output.glob("checkpoint-*"):
        if not re.fullmatch(r"checkpoint-\d+", path.name):
            continue
        if not all((path / name).is_file() and (path / name).stat().st_size for name in required):
            continue
        state = checkpoint_state(path)
        if state and state[0] == int(path.name.split("-")[-1]):
            found.append((state[0], state[1], path))
    return sorted(found)

def train_stage(config, task, retry_name):
    output = Path(config["output_dir"])
    config_file = drive_runs / f"{output.name}-config.json"
    if config_file.exists():
        if load_json(config_file) != config:
            raise ValueError(f"该阶段配置已改变，请增加 {retry_name} 编号。")
    else:
        save_json(config_file, config)
    if not (output / "training_report.json").is_file():
        args = ["-m", "yuki_model", "train", "--config", config_file]
        if output.exists() and any(output.iterdir()):
            checkpoints = complete_checkpoints(output)
            if not checkpoints:
                raise RuntimeError(f"上次训练中断且该阶段无法恢复，请增加 {retry_name} 编号。")
            checkpoint = checkpoints[-1][2]
            print("恢复检查点", checkpoint)
            args.extend(("--resume", checkpoint))
        run(*args, log=drive_runs / f"{output.name}.log")
    report = verify_run(output, task)
    checkpoints = complete_checkpoints(output)
    saved_epochs = [epoch for _, epoch, _ in checkpoints]
    expected_epochs = list(range(1, config["epochs"] + 1))
    if saved_epochs != expected_epochs:
        raise RuntimeError(f"云盘检查点不完整：预期 {expected_epochs}，实际 {saved_epochs}。")
    print("云盘检查点轮数", saved_epochs)
    print(task, "步数", report["steps"], "验证损失", report["before"]["eval_loss"],
          "→", report["after"]["eval_loss"])
    return report

knowledge_output = drive_runs / f"knowledge-{KNOWLEDGE_ATTEMPT:02d}"
persona_output = drive_runs / f"persona-{PERSONA_ATTEMPT:02d}"

if RUN_TRAINING:
    knowledge_config = training_config(
        "train-sft.json", knowledge_data, knowledge_output, sequence_length=1024,
        epochs=KNOWLEDGE_EPOCHS,
    )
    knowledge_report = train_stage(knowledge_config, "knowledge_sft", "KNOWLEDGE_ATTEMPT")
    persona_config = training_config(
        "train-persona-continued.json", persona_data, persona_output,
        data_manifest=str(persona_data / "manifest.json"), initial_adapter=str(knowledge_output / "adapter"),
        epochs=PERSONA_EPOCHS,
    )
    persona_report = train_stage(persona_config, "complete_persona", "PERSONA_ATTEMPT")
    if persona_report["data"]["initial_adapter_sha256"] != knowledge_report["adapter_sha256"]:
        raise RuntimeError("人格权重的训练来源与本轮知识权重不一致。")
    print("训练产物已保存在", drive_runs)
else:
    print("训练关闭，已完成数据准备。")

## 5. 导出桌面离线模型

将完成训练的人格 LoRA 合并到底座，导出 Q4_K_M GGUF。检查合并后的张量确实变化，并验证最终文件的大小、文件头和 SHA-256。转换中间文件在 Colab 工作盘生成，经过校验的 GGUF 和模型清单写入 Drive。

In [ ]:
# 5. 在本地合并与量化，再将校验后的候选 GGUF 保存到 Drive
EXPORT_GGUF = True  # 仅训练 LoRA 时可关闭

if EXPORT_GGUF:
    export_report = verify_run(persona_output, "complete_persona")
    export_path = drive_runs / f"{persona_output.name}-gguf"
    converter = root / "llama.cpp"
    converter_commit = "b3fa69955ea81c3b60e77e3c1e2224c4a826ffb9"

    if not export_path.exists():
        if shutil.disk_usage(root).free < 8 * 1024**3:
            raise RuntimeError("GGUF 合并与量化至少需要 8 GiB 本地可用空间。")
        if not (converter / ".git").is_dir():
            subprocess.run(["git", "init", str(converter)], check=True)
        subprocess.run(["git", "-C", str(converter), "fetch", "--depth=1",
                        "https://github.com/ggml-org/llama.cpp.git", converter_commit], check=True)
        subprocess.run(["git", "-C", str(converter), "checkout", "--detach", "FETCH_HEAD"], check=True)
        actual = subprocess.check_output(["git", "-C", str(converter), "rev-parse", "HEAD"], text=True).strip()
        if actual != converter_commit:
            raise RuntimeError("llama.cpp 转换器版本不匹配。")
        run("-m", "pip", "install", "numpy==2.2.6", "sentencepiece==0.2.1", "protobuf==4.25.8")
        build = converter / "build"
        subprocess.run(["cmake", "-S", str(converter), "-B", str(build),
                        "-DCMAKE_BUILD_TYPE=Release", "-DGGML_NATIVE=OFF",
                        "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_TOOLS=ON",
                        "-DLLAMA_BUILD_COMMON=ON", "-DLLAMA_BUILD_APP=OFF",
                        "-DLLAMA_BUILD_SERVER=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
                        "-DLLAMA_OPENSSL=OFF"], check=True)
        subprocess.run(["cmake", "--build", str(build), "--target", "llama-quantize", "--parallel", "2"], check=True)
        with tempfile.TemporaryDirectory(prefix="export-", dir=root) as work:
            work = Path(work)
            merged = work / "merged"
            candidate = work / "gguf"
            run("-m", "yuki_model", "merge", "--base", base, "--adapter", persona_output / "adapter",
                "--output", merged)
            merge_check = """
import sys
from safetensors import safe_open
import torch

key = "model.layers.0.self_attn.q_proj.weight"
with safe_open(sys.argv[1], framework="pt", device="cpu") as file:
    original = file.get_tensor(key)
with safe_open(sys.argv[2], framework="pt", device="cpu") as file:
    updated = file.get_tensor(key)
if original.shape != updated.shape or not torch.isfinite(updated).all():
    raise RuntimeError("Merged tensor is incompatible or non-finite")
changed = int(torch.count_nonzero(updated != original.to(updated.dtype)).item())
if changed == 0:
    raise RuntimeError("LoRA merge did not change the targeted tensor")
print(changed)
"""
            checked = subprocess.check_output(
                [str(python), "-c", merge_check, str(base / "model.safetensors"), str(merged / "model.safetensors")],
                text=True,
            ).strip()
            save_json(merged / "merge-validation.json", {
                "checkedTensor": "model.layers.0.self_attn.q_proj.weight",
                "changedElements": int(checked),
                "adapterSha256": export_report["adapter_sha256"],
            })
            run("scripts/export_gguf.py", "--llama-cpp", converter, "--merged", merged,
                "--training-report", persona_output / "training_report.json", "--base-license", base / "LICENSE",
                "--quantizer", build / "bin/llama-quantize", "--output", candidate,
                "--work-dir", work / "conversion", "--remove-fp16")
            manifest = load_json(candidate / "manifest.json")
            manifest["name"] = f"雪乃本地角色模型 · Qwen 1.5B · A100 · {RUN_ID}/{persona_output.name}"
            save_json(candidate / "manifest.json", manifest)
            publish_directory(candidate, export_path)

    manifest = load_json(export_path / "manifest.json")
    weights = export_path / manifest["file"]
    with weights.open("rb") as stream:
        if stream.read(4) != b"GGUF":
            raise RuntimeError("导出的文件缺少 GGUF 文件头。")
    if (manifest["training"]["adapterSha256"] != export_report["adapter_sha256"]
            or manifest["llamaCppConverterCommit"] != converter_commit
            or weights.stat().st_size != manifest["bytes"] or digest(weights) != manifest["sha256"]):
        raise RuntimeError("Drive 上的 GGUF 权重或来源校验失败。")
    print("候选 GGUF 已保存并通过文件校验，等待人工评估：", weights)
else:
    print("本次跳过 GGUF 导出。")

## 6. 对照评估角色表现与事实准确性

用同一人物提示和题集比较基础模型、知识阶段、最终模型及可用的旧模型。评测保存原始回答和人工评分表，重点核对雪乃的语气、多轮一致性、人物关系，以及遇到未知剧情时是否编造。

In [ ]:
# 6. 用同一人物提示和题集对照基础模型、旧版与本轮两阶段权重
from zipfile import ZipFile

if (persona_output / "training_report.json").is_file():
    verify_run(knowledge_output, "knowledge_sft")
    verify_run(persona_output, "complete_persona")
    old = root / "old-adapter"
    old_zip = root / "Yukino-Persona-LoRA-0.4.0.zip"
    old_url = "https://github.com/fqxxzyw/Yukino-Companion/releases/download/v0.4.0-local.13/" + old_zip.name
    expected_zip = "fff8bdcb2df7f0bfa1a30d2983ca1e7498f241ee978d41a206db54ba456989e5"
    old_adapter = None
    try:
        if not old_zip.is_file() or digest(old_zip) != expected_zip:
            subprocess.run(["curl", "-fL", "--retry", "2", "-o", str(old_zip), old_url], check=True)
        if digest(old_zip) != expected_zip:
            raise ValueError("旧 LoRA 归档 SHA-256 不符。")
        with ZipFile(old_zip) as archive:
            for name in ("adapter_config.json", "adapter_model.safetensors"):
                archive.extract("adapter/" + name, old)
        old_adapter = old / "adapter"
    except (OSError, ValueError, subprocess.CalledProcessError) as error:
        message = f"旧 LoRA 对照不可用，继续评估本轮模型：{error}"
        print(message)
        (drive_runs / "old-baseline-unavailable.txt").write_text(message + "\n", encoding="utf-8")
    baselines = [("base", None)]
    if old_adapter is not None:
        baselines.append(("old", old_adapter))
    baselines.extend((
        (knowledge_output.name, knowledge_output / "adapter"),
        (persona_output.name, persona_output / "adapter"),
    ))

    sets = {
        "held-out": persona_data / "test-prompts.jsonl",
        "challenge": challenge,
        "known-facts": knowledge_data / "known-fact-prompts.jsonl",
    }
    if PRIVATE_TEST_FILE:
        sets["private"] = data_root / PRIVATE_TEST_FILE
    for set_name, prompts in sets.items():
        prompt_hash = digest(prompts)
        for label, adapter in baselines:
            adapter_hash = digest(adapter / "adapter_model.safetensors")[:12] if adapter else "base"
            final = drive_runs / "evaluation" / f"{set_name}-{prompt_hash[:12]}-{label}-{adapter_hash}"
            if final.exists():
                print("已有报告", final)
                continue
            with tempfile.TemporaryDirectory(dir=root) as work:
                local = Path(work) / "result"
                args = ["scripts/evaluate_persona.py", "--device", "cuda", "--prompt-mode", "training",
                        "--prompts", prompts, "--output", local]
                if adapter:
                    args += ["--adapter", adapter]
                run(*args)
                result = json.loads((local / "report.json").read_text(encoding="utf-8"))
                if result["backend"]["device"] != "cuda" or result["backend"]["cpu_fallback_used"]:
                    raise RuntimeError("对照评估没有全程使用 CUDA。")
                final.parent.mkdir(exist_ok=True)
                publish_directory(local, final)
                print("保存报告", final)
else:
    print("完整试训完成后再运行评估。")

## 云盘目录和断点恢复

```text
Yukino-Companion/
├── yukino-a100-multiepoch.ipynb
├── A100-多轮训练说明.md
├── source/
│   └── training-source-e846aa4c-checkpoints-v2.zip
├── data/
│   ├── epubs/
│   ├── scenarios.jsonl
│   └── challenge-prompts.jsonl
└── retrain/epub-a100-v2/
    ├── inputs.json
    ├── setup-colab-cuda.log
    ├── environment-colab-cuda.log
    ├── prepared/
    │   ├── processed/knowledge.sqlite
    │   ├── sft/
    │   └── persona/
    ├── knowledge-01-config.json
    ├── knowledge-01.log
    ├── knowledge-01/
    │   ├── checkpoint-*/
    │   ├── adapter/
    │   └── training_report.json
    ├── persona-01-config.json
    ├── persona-01.log
    ├── persona-01/
    │   ├── checkpoint-*/
    │   ├── adapter/
    │   └── training_report.json
    ├── persona-01-gguf/
    │   ├── yukino-persona-q4_k_m.gguf
    │   └── manifest.json
    └── evaluation/
```

同一运行使用相同的轮数、输入数据和阶段编号续跑。每轮保存的目录有完整状态后才可用于恢复。修改训练轮数或语料时使用新 `RUN_ID`。训练中断时，最近一轮检查点之后尚未保存的步骤会重新执行。

原作事实回归用已知事实的改写。人格评估另外使用保留场景和挑战题。最终 GGUF 是待验收的候选模型，需查看原始回答和人工评分确认是否符合雪乃性格。

## 7. 双模型对话对比

默认比较旧发布版和当前笔记本训出的新版。下拉菜单还会显示已完成的知识阶段，以及 Drive 中已完成的另一轮人格训练。两边用同一份 Qwen 底座、同一个雪乃人物提示和固定解码设置，分别启用各自的 LoRA。

先运行 7.1，再运行 7.2。界面留在当前 Colab 页面，可以一行写一个问题，一次提交最多 12 道。选「每题独立」可对同一道题做直接对照。选「连续对话」时，两边各自保留最近四轮，适合观察追问时是否保持角色。勾选「保存到 Drive」才会写入提问和回答。

运行前需要完成第 1–4 节。第 5 节 GGUF 导出和第 6 节批量评估可以单独执行。这里展示的是 LoRA 权重的输出。

### 7.1 核对两份模型并准备推理

验证本轮训练报告和权重，复用第 6 节缓存的旧发布版。实际提问时才加载底座，之后重复提问会复用同一个推理进程。

In [ ]:
import json
import os
import select
import subprocess
from zipfile import ZipFile

if "persona_output" not in globals() or "python" not in globals():
    raise RuntimeError("请先运行第 1–4 节，读取云盘里已完成的训练。")

verify_run(persona_output, "complete_persona")
comparison_models = {}

def register_model(key, label, output, task):
    report = verify_run(output, task)
    comparison_models[key] = {
        "label": label,
        "path": str(output / "adapter"),
        "sha256": report["adapter_sha256"],
    }

register_model("current", f"本轮人格版 · {RUN_ID}", persona_output, "complete_persona")
register_model("knowledge", f"本轮知识版 · {RUN_ID}", knowledge_output, "knowledge_sft")

for alternate_id in ("epub-a100-v1", "epub-a100-v2"):
    if alternate_id == RUN_ID:
        continue
    alternate = project / "retrain" / alternate_id / "persona-01"
    if (alternate / "training_report.json").is_file():
        try:
            register_model(alternate_id, f"人格版 · {alternate_id}", alternate, "complete_persona")
        except (OSError, KeyError, ValueError, RuntimeError) as error:
            print(f"跳过 {alternate_id}，校验未通过：{error}")

old_zip = root / "Yukino-Persona-LoRA-0.4.0.zip"
old_sha256 = "fff8bdcb2df7f0bfa1a30d2983ca1e7498f241ee978d41a206db54ba456989e5"
old_weights_sha256 = "21f347ebc4b8658705dfd601270955708814d37a18bf5272cd0bfa1878cfdb7b"
try:
    if not old_zip.is_file() or digest(old_zip) != old_sha256:
        old_url = "https://github.com/fqxxzyw/Yukino-Companion/releases/download/v0.4.0-local.13/" + old_zip.name
        subprocess.run(["curl", "-fL", "--retry", "2", "--max-time", "90",
                        "-o", str(old_zip), old_url], check=True)
    if digest(old_zip) != old_sha256:
        raise RuntimeError("旧发布版归档 SHA-256 不符。")
    old_dir = root / "old-adapter"
    with ZipFile(old_zip) as archive:
        for name in ("adapter_config.json", "adapter_model.safetensors"):
            archive.extract("adapter/" + name, old_dir)
    old_adapter = old_dir / "adapter"
    if digest(old_adapter / "adapter_model.safetensors") != old_weights_sha256:
        raise RuntimeError("旧发布版适配器 SHA-256 不符。")
    comparison_models["release"] = {
        "label": "旧发布版 · 人格 LoRA v1.2",
        "path": str(old_adapter),
        "sha256": old_weights_sha256,
    }
except (OSError, KeyError, ValueError, RuntimeError, subprocess.CalledProcessError) as error:
    print(f"旧发布版暂不可用，仍可选择本轮知识版和人格版：{error}")

COMPARE_WORKER = r'''
from contextlib import redirect_stdout
import json
from pathlib import Path
import sys

with redirect_stdout(sys.stderr):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from yuki_model.persona import load_character

base = Path(sys.argv[1])
character = Path(sys.argv[2])
model = tokenizer = system = None

def reply(item):
    global model, tokenizer, system
    if model is None:
        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
            raise RuntimeError("对比需要支持 BF16 的 CUDA GPU。")
        torch.cuda.set_device(0)
        system, _ = load_character(character, "training")
        tokenizer = AutoTokenizer.from_pretrained(base, local_files_only=True, trust_remote_code=False)
        backbone = AutoModelForCausalLM.from_pretrained(
            base, torch_dtype=torch.bfloat16, local_files_only=True, trust_remote_code=False,
        ).to("cuda:0")
        model = PeftModel.from_pretrained(backbone, item["path"], adapter_name=item["key"],
                                         is_trainable=False, local_files_only=True)
    elif item["key"] not in model.peft_config:
        model.load_adapter(item["path"], adapter_name=item["key"], is_trainable=False, local_files_only=True)
    model.set_adapter(item["key"])
    model.eval()
    messages = [{"role": "system", "content": system}, *item["messages"]]
    while True:
        encoded = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        )
        length = encoded["input_ids"].shape[-1]
        if length <= 8192 - 192:
            break
        if len(messages) <= 2:
            raise ValueError("单题太长，请缩短后再试。")
        del messages[1:3]
    encoded = encoded.to("cuda:0")
    with torch.inference_mode():
        tokens = model.generate(**encoded, max_new_tokens=192, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(tokens[0, length:], skip_special_tokens=True).strip()

for line in sys.stdin:
    try:
        with redirect_stdout(sys.stderr):
            request = json.loads(line)
            answers = [reply(item) for item in request]
        response = {"answers": answers}
    except Exception as error:
        response = {"error": f"{type(error).__name__}: {error}"}
    print(json.dumps(response, ensure_ascii=False), flush=True)
'''

if "stop_comparison_worker" in globals():
    stop_comparison_worker()
comparison_worker = None
comparison_log = drive_runs / "comparison-runtime.log"

def stop_comparison_worker():
    global comparison_worker
    worker = comparison_worker
    comparison_worker = None
    if worker is None:
        return
    if worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=5)
        except subprocess.TimeoutExpired:
            worker.kill()
            worker.wait()
    worker.stdin.close()
    worker.stdout.close()

def compare_request(items):
    global comparison_worker
    if comparison_worker is None or comparison_worker.poll() is not None:
        stop_comparison_worker()
        with comparison_log.open("a", encoding="utf-8") as log:
            comparison_worker = subprocess.Popen(
                [str(python), "-u", "-c", COMPARE_WORKER, str(base), str(repo / "characters/yukino")],
                cwd=service, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=log,
                text=True, encoding="utf-8", bufsize=1,
                env=dict(os.environ, USE_TF="0", USE_TORCH="1"),
            )
    worker = comparison_worker
    try:
        worker.stdin.write(json.dumps(items, ensure_ascii=False) + "\n")
        worker.stdin.flush()
        if not select.select([worker.stdout], [], [], 300)[0]:
            raise TimeoutError("推理等待超过 5 分钟。")
        line = worker.stdout.readline()
        if not line:
            raise RuntimeError(f"推理进程退出，日志见 {comparison_log}")
        response = json.loads(line)
    except (OSError, ValueError, RuntimeError) as error:
        stop_comparison_worker()
        raise RuntimeError(f"{error}。可再次点击比较，日志见 {comparison_log}") from error
    if "error" in response:
        raise RuntimeError(response["error"])
    return response["answers"]

print("可选模型", "、".join(item["label"] for item in comparison_models.values()))


### 7.2 打开并排问答界面

选择左右模型、输入问题并点击「比较回答」。切换模型或对话模式后从新历史开始。点击「释放模型」可回收这节占用的 GPU 显存。

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

import ipywidgets as widgets
from IPython.display import display

if "compare_request" not in globals():
    raise RuntimeError("请先运行 7.1。")

model_options = [(item["label"], key) for key, item in comparison_models.items()]
left = widgets.Dropdown(options=model_options, value="release" if "release" in comparison_models else "knowledge",
                        description="左栏")
right = widgets.Dropdown(options=model_options, value="current", description="右栏")
mode = widgets.ToggleButtons(options=[("每题独立", "single"), ("连续对话", "chat")])
questions = widgets.Textarea(placeholder="每行一个问题，最多 12 行", layout=widgets.Layout(width="100%", height="108px"))
ask = widgets.Button(description="比较回答", button_style="primary")
clear = widgets.Button(description="清空对话")
release_gpu = widgets.Button(description="释放模型")
save_results = widgets.Checkbox(value=False, description="保存到 Drive", indent=False)
status = widgets.Label(value="请输入问题，首次提问时加载模型。")
answers = [widgets.Textarea(disabled=True, layout=widgets.Layout(width="100%", height="360px")) for _ in range(2)]
title = [widgets.Label(), widgets.Label()]
history = [[], []]
saved_file = drive_runs / "comparison" / f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid4().hex[:6]}.jsonl"

def reset_comparison(_=None):
    history[0].clear()
    history[1].clear()
    for index, key in enumerate((left.value, right.value)):
        title[index].value = comparison_models[key]["label"]
        answers[index].value = ""
    status.value = "已清空对话。"

def release_comparison(_):
    stop_comparison_worker()
    status.value = "已释放模型，再次提问会重新加载。"

def submit_comparison(_):
    selected = (left.value, right.value)
    batch = [line.strip() for line in questions.value.splitlines() if line.strip()]
    if selected[0] == selected[1]:
        status.value = "请选择两个不同的模型。"
        return
    if not 1 <= len(batch) <= 12 or any(len(question) > 3000 for question in batch):
        status.value = "每次输入 1–12 道题，每题最多 3,000 字。"
        return
    controls = (left, right, mode, questions, ask, clear, release_gpu, save_results)
    for control in controls:
        control.disabled = True
    for box in answers:
        box.value = ""
    try:
        for number, question in enumerate(batch, 1):
            status.value = f"正在回答第 {number}/{len(batch)} 题。"
            user = {"role": "user", "content": question}
            items = [
                {"key": key, "path": comparison_models[key]["path"],
                 "messages": [*(history[index] if mode.value == "chat" else []), user]}
                for index, key in enumerate(selected)
            ]
            pair = compare_request(items)
            for index, answer in enumerate(pair):
                block = f"{number}. {question}\n\n{answer or '（模型未返回文字）'}"
                answers[index].value += ("\n\n" if answers[index].value else "") + block
                if mode.value == "chat":
                    history[index][:] = [*history[index], user, {"role": "assistant", "content": answer}][-8:]
            if save_results.value:
                saved_file.parent.mkdir(parents=True, exist_ok=True)
                record = {
                    "time": datetime.now(timezone.utc).isoformat(),
                    "mode": mode.value, "question": question,
                    "answers": [
                        {"model": comparison_models[key]["label"],
                         "adapter_sha256": comparison_models[key]["sha256"], "answer": answer}
                        for key, answer in zip(selected, pair)
                    ],
                }
                with saved_file.open("a", encoding="utf-8") as stream:
                    stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        status.value = f"已回答 {len(batch)} 道题。" + (f" 已保存：{saved_file.name}" if save_results.value else "")
    except (OSError, RuntimeError, ValueError) as error:
        status.value = f"第 {number} 题出错：{error}"
    finally:
        for control in controls:
            control.disabled = False

for control in (left, right, mode):
    control.observe(reset_comparison, names="value")
ask.on_click(submit_comparison)
clear.on_click(reset_comparison)
release_gpu.on_click(release_comparison)
reset_comparison()
status.value = "请输入问题，首次提问时加载模型。"
columns = [widgets.VBox([title[index], answers[index]], layout=widgets.Layout(width="49%"))
           for index in range(2)]
display(widgets.VBox([
    widgets.HBox([left, right]), mode, questions,
    widgets.HBox([ask, clear, release_gpu, save_results]), status,
    widgets.HBox(columns, layout=widgets.Layout(width="100%", justify_content="space-between")),
]))
